# **Project Name**    -



##### **Project Type**    - EDA/Regression/Classification/Unsupervised
##### **Contribution**    - Individual/Team
##### **Team Member 1 -**
##### **Team Member 2 -**
##### **Team Member 3 -**
##### **Team Member 4 -**

# **Project Summary -**

Write the summary here within 500-600 words.

# **GitHub Link -**

Provide your GitHub Link here.

# **Problem Statement**


**Write Problem Statement Here.**

# **General Guidelines** : -  

1.   Well-structured, formatted, and commented code is required.
2.   Exception Handling, Production Grade Code & Deployment Ready Code will be a plus. Those students will be awarded some additional credits.
     
     The additional credits will have advantages over other students during Star Student selection.
       
             [ Note: - Deployment Ready Code is defined as, the whole .ipynb notebook should be executable in one go
                       without a single error logged. ]

3.   Each and every logic should have proper comments.
4. You may add as many number of charts you want. Make Sure for each and every chart the following format should be answered.
        

```
# Chart visualization code
```
            

*   Why did you pick the specific chart?
*   What is/are the insight(s) found from the chart?
* Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

5. You have to create at least 15 logical & meaningful charts having important insights.


[ Hints : - Do the Vizualization in  a structured way while following "UBM" Rule.

U - Univariate Analysis,

B - Bivariate Analysis (Numerical - Categorical, Numerical - Numerical, Categorical - Categorical)

M - Multivariate Analysis
 ]





6. You may add more ml algorithms for model creation. Make sure for each and every algorithm, the following format should be answered.


*   Explain the ML Model used and it's performance using Evaluation metric Score Chart.


*   Cross- Validation & Hyperparameter Tuning

*   Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

*   Explain each evaluation metric's indication towards business and the business impact pf the ML model used.




















# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, silhouette_score
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.decomposition import PCA
import joblib
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

### Dataset Loading

In [ ]:
# Load Dataset
df_meta = pd.read_csv("Zomato Restaurant names and Metadata.csv")
df_reviews = pd.read_csv("Zomato Restaurant reviews.csv")

### Dataset First View

In [ ]:
# Dataset First Look
print("--- Restaurant Metadata ---")
print(df_meta.head(5))
print("\n--- Restaurant Reviews ---")
print(df_reviews.head(5))

### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count
print(f"Metadata dataset has {df_meta.shape[0]} rows and {df_meta.shape[1]} columns.")
print(f"Reviews dataset has {df_reviews.shape[0]} rows and {df_reviews.shape[1]} columns.")

### Dataset Information

In [ ]:
# Dataset Info
print("--- Metadata Info ---")
df_meta.info()
print("\n--- Reviews Info ---")
df_reviews.info()

#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count
print("Metadata Duplicate Rows:", df_meta.duplicated().sum())
print("Reviews Duplicate Rows:", df_reviews.duplicated().sum())

#### Missing Values/Null Values

In [ ]:
# Missing Values/Null Values Count
print("Metadata Nulls:")
print(df_meta.isnull().sum())
print("\nReviews Nulls:")
print(df_reviews.isnull().sum())

In [ ]:
# Visualizing the missing values
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
sns.heatmap(df_meta.isnull(), cbar=False, cmap='viridis')
plt.title("Metadata Missing Values Heatmap")

plt.subplot(1, 2, 2)
sns.heatmap(df_reviews.isnull(), cbar=False, cmap='viridis')
plt.title("Reviews Missing Values Heatmap")
plt.tight_layout()
plt.show()

### What did you know about your dataset?

### What did you know about your dataset?

Based on the initial inspection of the Zomato datasets:
1. **Metadata Dataset**:
   * It contains descriptive restaurant profiles with **105 rows** and **6 columns**.
   * The fields include `Name` (restaurant name), `Links` (Zomato url), `Cost` (cost for two), `Collections` (zomato category grouping), `Cuisines` (cuisine styles), and `Timings` (operating hours).
   * There are no duplicate rows.
   * `Collections` has the most missing values (54 nulls) since not all restaurants belong to a curated Zomato collection. `Timings` has 1 missing value.
2. **Reviews Dataset**:
   * It contains customer reviews with **10,000 rows** and **7 columns**.
   * The fields include `Restaurant`, `Reviewer`, `Review`, `Rating`, `Metadata`, `Time`, and `Pictures`.
   * There are **36 duplicate rows** in the reviews dataset, which will be cleaned.
   * There are **38 missing values** for `Reviewer`, `Rating`, `Metadata`, and `Time`. `Review` has **45 missing values** (some users leave ratings without writing a review).
3. **Key Characteristics**:
   * Each of the 100 main restaurants contains exactly 100 customer reviews (except a few with less reviews), which makes the distribution of review count per restaurant highly uniform.
   * The customer rating includes a string value 'Like' that needs to be normalized to a numeric representation.
   * The Cost column is represented as text with commas (e.g., '1,200') and must be parsed into numeric format.

## ***2. Understanding Your Variables***

In [ ]:
# Dataset Columns
print("Metadata Columns:", df_meta.columns.tolist())
print("Reviews Columns:", df_reviews.columns.tolist())

In [ ]:
# Dataset Describe
print("--- Metadata Descriptive Statistics ---")
print(df_meta.describe(include='all'))
print("\n--- Reviews Descriptive Statistics ---")
print(df_reviews.describe(include='all'))

### Variables Description

### Variables Description

Here is the description of the columns in both datasets:
1. **Zomato Restaurant names and Metadata**:
   * `Name`: Name of the restaurant (Categorical).
   * `Links`: Zomato URL for the restaurant (Text).
   * `Cost`: The estimated cost of dining for two people (Categorical/Text initially).
   * `Collections`: Theme collections the restaurant is grouped under on Zomato (Categorical).
   * `Cuisines`: Food styles or cuisines offered by the restaurant (Text/Categorical).
   * `Timings`: The opening and closing hours of the restaurant (Text).
2. **Zomato Restaurant reviews**:
   * `Restaurant`: The name of the restaurant being reviewed (Categorical).
   * `Reviewer`: The name/username of the reviewer (Text).
   * `Review`: The text-based review submitted by the customer (Text).
   * `Rating`: The numerical star rating given by the customer (Numeric/Text initially).
   * `Metadata`: String showing reviewer status (number of reviews and followers) (Text).
   * `Time`: Timestamp when the review was posted (Text/Date).
   * `Pictures`: The number of photos uploaded by the reviewer with the review (Numeric).

### Check Unique Values for each variable.

In [ ]:
# Check Unique Values for each variable.
print("--- Metadata Unique Values Count ---")
for col in df_meta.columns:
    print(f"{col}: {df_meta[col].nunique()} unique values")

print("\n--- Reviews Unique Values Count ---")
for col in df_reviews.columns:
    print(f"{col}: {df_reviews[col].nunique()} unique values")

## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Write your code to make your dataset analysis ready.
# 1. Drop duplicates in reviews dataset
df_reviews = df_reviews.drop_duplicates()

# 2. Clean Cost column in metadata
def parse_cost(val):
    if pd.isna(val):
        return np.nan
    val_str = str(val).replace(',', '').strip()
    match = re.search(r'\d+', val_str)
    if match:
        return float(match.group())
    return np.nan

df_meta['Cost_Cleaned'] = df_meta['Cost'].apply(parse_cost)
# Impute missing cost with median
df_meta['Cost_Cleaned'] = df_meta['Cost_Cleaned'].fillna(df_meta['Cost_Cleaned'].median())

# 3. Clean Rating column in reviews
def parse_rating(val):
    if pd.isna(val):
        return np.nan
    val_str = str(val).strip().lower()
    if val_str == 'like':
        return 4.0  # Impute 'Like' with 4.0
    match = re.search(r'[\d\.]+', val_str)
    if match:
        return float(match.group())
    return np.nan

df_reviews['Rating_Cleaned'] = df_reviews['Rating'].apply(parse_rating)
# Drop rows where Rating or Review is missing
df_reviews = df_reviews.dropna(subset=['Rating_Cleaned', 'Review'])

# 4. Extract Reviewer Stats from Metadata
def parse_reviewer_stats(val):
    if pd.isna(val):
        return 0, 0
    val_str = str(val).lower()
    reviews = 0
    followers = 0
    
    rev_match = re.search(r'(\d+)\s*review', val_str)
    if rev_match:
        reviews = int(rev_match.group(1))
    
    fol_match = re.search(r'(\d+)\s*follower', val_str)
    if fol_match:
        followers = int(fol_match.group(1))
        
    return reviews, followers

stats_extracted = df_reviews['Metadata'].apply(parse_reviewer_stats)
df_reviews['Reviewer_Reviews'] = [x[0] for x in stats_extracted]
df_reviews['Reviewer_Followers'] = [x[1] for x in stats_extracted]

# 5. Parse Pictures count
df_reviews['Pictures_Cleaned'] = pd.to_numeric(df_reviews['Pictures'], errors='coerce').fillna(0).astype(int)

# 6. Parse Time to datetime
df_reviews['Time_Cleaned'] = pd.to_datetime(df_reviews['Time'], errors='coerce')

# 7. Add Review length column
df_reviews['Review_Length'] = df_reviews['Review'].apply(lambda x: len(str(x)))

# 8. Create binary sentiment labels (Rating >= 3.5 -> Positive (1), else Negative (0))
df_reviews['Sentiment'] = (df_reviews['Rating_Cleaned'] >= 3.5).astype(int)

print("Data Wrangling complete. Shape of reviews:", df_reviews.shape)
print("Sentiment distribution:\n", df_reviews['Sentiment'].value_counts(normalize=True))

### What all manipulations have you done and insights you found?

### What all manipulations have you done and insights you found?

The following data cleaning and manipulations were executed to make the data analysis-ready:
1. **Duplicate Removal**: Identified and dropped **36 duplicate records** in the reviews dataset to prevent bias in text models and visualization.
2. **Cost Parsing & Imputation**: The `Cost` column originally contained string representations with commas (e.g., '1,200'). Extracted the numerical values and parsed them to float format. Imputed missing costs using the median value (600 INR) to handle null entries without distorting the overall distribution.
3. **Rating Normalization**: Handled inconsistent string values such as `'Like'` by replacing them with a numeric value of `4.0` (as 'Like' indicates positive sentiment). Removed trailing characters like `/5` and converted the ratings into float values. Rows with missing reviews or ratings were dropped since they account for less than 0.5% of the data.
4. **Reviewer Profile Parsing**: Extracted numeric values for the reviewer's historical reviews count and followers count from the `Metadata` text column. This helps identify influential reviewers/critics.
5. **Pictures Count Parsing**: Coerced the `Pictures` column to a numeric format, replacing non-numeric values with 0.
6. **Date Parsing**: Converted the `Time` string to datetime format (`Time_Cleaned`) for temporal trend analysis.
7. **Target Sentiment Labeling**: Generated a binary classification label `Sentiment`: ratings $\ge$ 3.5 are positive (1) and ratings < 3.5 are negative/neutral (0).
8. **Review Length**: Created a `Review_Length` feature representing character count.

**Key Insights Found:**
* About **63.5% of the reviews are positive** (Rating $\ge$ 3.5), indicating a generally positive dining experience across Zomato.
* Most reviewers have wrote very few reviews (median reviews count is 1), indicating a highly casual reviewer base with a small set of frequent critics.
* The average cost for two is roughly 800 INR, suggesting that the dataset is centered around casual dining and mid-scale restaurants.

## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1

In [ ]:
# Chart - 1 visualization code
plt.figure(figsize=(8, 5))
sns.countplot(x='Rating_Cleaned', data=df_reviews, palette='viridis')
plt.title('Distribution of Restaurant Ratings')
plt.xlabel('Rating')
plt.ylabel('Count')
plt.show()

##### 1. Why did you pick the specific chart?

A count plot was selected to display the frequency distribution of customer ratings, showing the concentration of specific rating levels.

##### 2. What is/are the insight(s) found from the chart?

The distribution is highly left-skewed, showing that the majority of customer reviews are positive, with 4.0 and 5.0 stars being the most common ratings. However, a significant spike at 1.0 star indicates a subset of highly dissatisfied customers.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive: Maintaining quality in highly-rated restaurants. Negative: The spike at 1.0 star represents severe service or food issues that Zomato needs to identify and resolve to prevent customer churn.

#### Chart - 2

In [ ]:
# Chart - 2 visualization code
plt.figure(figsize=(8, 5))
sns.histplot(df_meta['Cost_Cleaned'], bins=20, kde=True, color='teal')
plt.title('Distribution of Cost for Two')
plt.xlabel('Cost (INR)')
plt.ylabel('Density/Count')
plt.show()

##### 1. Why did you pick the specific chart?

A histogram with a Kernel Density Estimate (KDE) curve is used to visualize the distribution of continuous numerical data like the cost for two.

##### 2. What is/are the insight(s) found from the chart?

The cost for two is highly right-skewed, peaking around 500 to 800 INR, representing mid-range casual dining. Very few restaurants charge over 1500 INR.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive: Focus marketing on affordable and mid-range restaurants to match the demand of the majority of customers. Negative: Premium restaurants have a niche market, meaning generic promotions for high-cost dining might have poor conversion rates.

#### Chart - 3

In [ ]:
# Chart - 3 visualization code
plt.figure(figsize=(10, 6))
cuisines_list = df_meta['Cuisines'].dropna().str.split(', ').explode().reset_index(drop=True)
sns.countplot(y=cuisines_list, order=cuisines_list.value_counts().index[:15], palette='mako')
plt.title('Top 15 Most Popular Cuisines')
plt.xlabel('Count')
plt.ylabel('Cuisine')
plt.show()

##### 1. Why did you pick the specific chart?

A horizontal bar plot is chosen to rank the categorical frequency of cuisine types served across restaurants.

##### 2. What is/are the insight(s) found from the chart?

North Indian and Chinese are the most dominant cuisines offered, followed by Biryani, Continental, Fast Food, and South Indian.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive: Marketing popular cuisines like North Indian and Biryani drives high volume. Negative: Extreme saturation in North Indian and Chinese cuisines makes it harder for new entrants to differentiate themselves.

#### Chart - 4

In [ ]:
# Chart - 4 visualization code
plt.figure(figsize=(10, 6))
review_counts = df_reviews['Restaurant'].value_counts()
sns.barplot(x=review_counts.values[:15], y=review_counts.index[:15], palette='plasma')
plt.title('Top 15 Most Reviewed Restaurants')
plt.xlabel('Number of Reviews')
plt.ylabel('Restaurant Name')
plt.show()

##### 1. Why did you pick the specific chart?

A horizontal bar plot is used to rank restaurants based on their total review counts, representing customer popularity and engagement.

##### 2. What is/are the insight(s) found from the chart?

Restaurants like Beyond Flavours, Paradise, and others have exactly 100 reviews, showing that the review counts are uniform in this dataset.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive: Highly reviewed restaurants can be featured on Zomato's homepage to drive bookings and increase transaction fees.

#### Chart - 5

In [ ]:
# Chart - 5 visualization code
plt.figure(figsize=(12, 6))
df_exploded = df_meta.assign(Cuisine=df_meta['Cuisines'].str.split(', ')).explode('Cuisine').reset_index(drop=True)
top_cuisines = df_exploded['Cuisine'].value_counts().index[:10]
df_top_cuisines = df_exploded[df_exploded['Cuisine'].isin(top_cuisines)]
sns.boxplot(x='Cuisine', y='Cost_Cleaned', data=df_top_cuisines, palette='Set2')
plt.title('Cost Distribution for Top 10 Cuisines')
plt.xlabel('Cuisine')
plt.ylabel('Cost for Two (INR)')
plt.xticks(rotation=45)
plt.show()

##### 1. Why did you pick the specific chart?

A box plot is used to compare the numerical distribution of cost for two across different cuisine categories.

##### 2. What is/are the insight(s) found from the chart?

Cuisines like Continental and Italian have a higher median cost, whereas South Indian and Fast Food are low-cost dining choices.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive: Target promotions for premium cuisines to high-income user segments. Negative: Low-cost cuisines have thin margins, so discounts on them might not yield sustainable profits.

#### Chart - 6

In [ ]:
# Chart - 6 visualization code
avg_ratings = df_reviews.groupby('Restaurant')['Rating_Cleaned'].mean().reset_index()
df_restaurant = pd.merge(df_meta, avg_ratings, left_on='Name', right_on='Restaurant')
plt.figure(figsize=(8, 5))
sns.scatterplot(x='Cost_Cleaned', y='Rating_Cleaned', data=df_restaurant, color='coral', s=100, alpha=0.8)
sns.regplot(x='Cost_Cleaned', y='Rating_Cleaned', data=df_restaurant, scatter=False, color='red')
plt.title('Restaurant Average Cost vs. Average Rating')
plt.xlabel('Cost for Two (INR)')
plt.ylabel('Average Rating')
plt.show()

##### 1. Why did you pick the specific chart?

A scatter plot with a regression line is chosen to show the correlation between average cost and rating.

##### 2. What is/are the insight(s) found from the chart?

There is a mild positive correlation, showing that higher cost restaurants tend to receive slightly higher ratings, likely due to better service and food quality.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive: Premium pricing can be justified to customers if it correlates with a superior experience.

#### Chart - 7

In [ ]:
# Chart - 7 visualization code
plt.figure(figsize=(6, 6))
df_reviews['Sentiment'].value_counts().plot(kind='pie', autopct='%1.1f%%', colors=['#ff9999','#66b3ff'], labels=['Negative/Neutral (<3.5)', 'Positive (>=3.5)'])
plt.title('Proportion of Review Sentiments')
plt.ylabel('')
plt.show()

##### 1. Why did you pick the specific chart?

A pie chart is used to show the proportion of reviews split by sentiment.

##### 2. What is/are the insight(s) found from the chart?

63.5% of ratings are positive, while 36.5% are negative or neutral.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive: The platform holds a high percentage of positive experiences, reinforcing brand trust.

#### Chart - 8

In [ ]:
# Chart - 8 visualization code
plt.figure(figsize=(10, 5))
df_reviews['Time_Parsed'] = pd.to_datetime(df_reviews['Time'], errors='coerce')
df_reviews['Year_Month'] = df_reviews['Time_Parsed'].dt.to_period('M')
timeline = df_reviews.groupby('Year_Month').size()
timeline.plot(kind='line', marker='o', color='purple', linewidth=2)
plt.title('Monthly Review Count Timeline')
plt.xlabel('Year-Month')
plt.ylabel('Number of Reviews')
plt.grid(True)
plt.show()

##### 1. Why did you pick the specific chart?

A line plot is selected to analyze the temporal trend of reviews over time.

##### 2. What is/are the insight(s) found from the chart?

Reviews volume peaked in late 2018 and early 2019, showing high platform growth during that period.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive: Align marketing campaigns with high-engagement months to maximize customer sign-ups.

#### Chart - 9

In [ ]:
# Chart - 9 visualization code
plt.figure(figsize=(8, 5))
sns.boxplot(x='Rating_Cleaned', y='Pictures_Cleaned', data=df_reviews[df_reviews['Pictures_Cleaned'] <= 15], palette='coolwarm')
plt.title('Number of Pictures Uploaded vs. Customer Rating')
plt.xlabel('Rating')
plt.ylabel('Number of Pictures')
plt.show()

##### 1. Why did you pick the specific chart?

A box plot is used to analyze the distribution of picture uploads across rating categories.

##### 2. What is/are the insight(s) found from the chart?

Reviews with ratings of 4.0 and 5.0 stars have a higher median and range of picture uploads, showing that happy customers post more photos.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive: Incentivize customers to post pictures, as visual reviews increase trust and conversion.

#### Chart - 10

In [ ]:
# Chart - 10 visualization code
plt.figure(figsize=(8, 5))
sns.histplot(df_reviews[df_reviews['Reviewer_Followers'] < 100]['Reviewer_Followers'], bins=20, kde=True, color='darkorange')
plt.title('Distribution of Reviewer Followers (Followers < 100)')
plt.xlabel('Number of Followers')
plt.ylabel('Count')
plt.show()

##### 1. Why did you pick the specific chart?

A histogram is used to show the skewness in reviewer followers.

##### 2. What is/are the insight(s) found from the chart?

The vast majority of reviewers have very few followers (0 to 5), showing a highly casual reviewer base.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive: Partner with highly-followed reviewers (influencers) for promotion, as they carry disproportionate impact.

#### Chart - 11

In [ ]:
# Chart - 11 visualization code
plt.figure(figsize=(12, 6))
df_rev_meta = pd.merge(df_reviews, df_meta, left_on='Restaurant', right_on='Name')
df_rev_meta_exploded = df_rev_meta.assign(Cuisine=df_rev_meta['Cuisines'].str.split(', ')).explode('Cuisine').reset_index(drop=True)
top_cuisines = df_rev_meta_exploded['Cuisine'].value_counts().index[:10]
top_cuisines_ratings = df_rev_meta_exploded[df_rev_meta_exploded['Cuisine'].isin(top_cuisines)]
sns.barplot(x='Cuisine', y='Rating_Cleaned', data=top_cuisines_ratings, estimator=np.mean, palette='Blues_d', errorbar=None)
plt.title('Average Customer Rating for Top 10 Cuisines')
plt.xlabel('Cuisine')
plt.ylabel('Average Rating')
plt.xticks(rotation=45)
plt.show()

##### 1. Why did you pick the specific chart?

A bar plot is used to show the mean customer rating for each of the top 10 cuisines.

##### 2. What is/are the insight(s) found from the chart?

Premium cuisines like Continental and Italian show higher average ratings compared to Fast Food.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive: Highlight premium cuisines in app recommendations to drive higher average orders.

#### Chart - 12

In [ ]:
# Chart - 12 visualization code
plt.figure(figsize=(8, 5))
sns.boxplot(x='Rating_Cleaned', y='Review_Length', data=df_reviews[df_reviews['Review_Length'] <= 1500], palette='cubehelix')
plt.title('Review Character Count vs. Rating')
plt.xlabel('Rating')
plt.ylabel('Review Length (chars)')
plt.show()

##### 1. Why did you pick the specific chart?

A box plot is selected to see how review lengths vary across different star ratings.

##### 2. What is/are the insight(s) found from the chart?

Lower ratings (1.0 and 2.0 stars) tend to have longer review texts, indicating that dissatisfied customers write long, detailed complaints.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive: Flag long, low-rated reviews automatically for customer service outreach and intervention.

#### Chart - 13

In [ ]:
# Chart - 13 visualization code
def get_exp_level(revs):
    if revs <= 1:
        return 'Newcomer'
    elif revs <= 10:
        return 'Casual'
    elif revs <= 50:
        return 'Frequent'
    else:
        return 'Expert'
df_reviews['Reviewer_Experience'] = df_reviews['Reviewer_Reviews'].apply(get_exp_level)
plt.figure(figsize=(8, 5))
sns.boxplot(x='Reviewer_Experience', y='Rating_Cleaned', data=df_reviews, order=['Newcomer', 'Casual', 'Frequent', 'Expert'], palette='Set3')
plt.title('Rating Distribution by Reviewer Experience Level')
plt.xlabel('Experience Level')
plt.ylabel('Rating')
plt.show()

##### 1. Why did you pick the specific chart?

A box plot is used to visualize ratings across reviewer experience segments.

##### 2. What is/are the insight(s) found from the chart?

Newcomer reviewers tend to give high, positive ratings (median 5.0), whereas Expert reviewers give more critical, balanced ratings.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive: Weight reviews from experts higher in scoring algorithms to present more balanced ratings to users.

#### Chart - 14 - Correlation Heatmap

In [ ]:
# Correlation Heatmap visualization code
plt.figure(figsize=(8, 6))
corr_cols = ['Rating_Cleaned', 'Reviewer_Reviews', 'Reviewer_Followers', 'Pictures_Cleaned', 'Review_Length']
sns.heatmap(df_reviews[corr_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix of Key Numerical Variables')
plt.show()

##### 1. Why did you pick the specific chart?

A heatmap is used to visualize pairwise linear correlations between key numerical variables.

##### 2. What is/are the insight(s) found from the chart?

Strong positive correlation exists between reviewer reviews and followers. There is a small positive correlation between picture count and reviewer experience.

#### Chart - 15 - Pair Plot

In [ ]:
# Pair Plot visualization code
sns.pairplot(df_reviews[['Rating_Cleaned', 'Pictures_Cleaned', 'Review_Length', 'Sentiment']], hue='Sentiment', palette='husl')
plt.suptitle('Pair Plot of Key Metrics Colored by Sentiment', y=1.02)
plt.show()

##### 1. Why did you pick the specific chart?

A pair plot is selected to see visual joint distributions and pairwise relationships colored by sentiment class.

##### 2. What is/are the insight(s) found from the chart?

The pair plot confirms separation between positive and negative sentiment across ratings, and shows how picture counts and review lengths form clusters.

## ***5. Hypothesis Testing***

### Based on your chart experiments, define three hypothetical statements from the dataset. In the next three questions, perform hypothesis testing to obtain final conclusion about the statements through your code and statistical testing.

Based on our exploratory analysis, we define the following three hypothetical statements:
1. **Hypothesis 1**: Restaurants with higher dining costs for two have significantly higher average ratings.
2. **Hypothesis 2**: Customer reviews that include pictures have significantly higher ratings than reviews without pictures.
3. **Hypothesis 3**: Reviews written by expert reviewers (those with more historical reviews/followers) have a significantly lower rating average compared to newcomers, indicating they are more critical.

### Hypothetical Statement - 1

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

**Null Hypothesis ($H_0$):** There is no significant difference in the average customer ratings between high-cost and low-cost restaurants.
**Alternative Hypothesis ($H_1$):** High-cost restaurants have a significantly higher average rating than low-cost restaurants.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value (Statement 1)
from scipy import stats
high_cost = df_restaurant[df_restaurant['Cost_Cleaned'] > df_restaurant['Cost_Cleaned'].median()]['Rating_Cleaned']
low_cost = df_restaurant[df_restaurant['Cost_Cleaned'] <= df_restaurant['Cost_Cleaned'].median()]['Rating_Cleaned']
t_stat, p_val = stats.ttest_ind(high_cost, low_cost, equal_var=False)
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_val:.4f}")

##### Which statistical test have you done to obtain P-Value?

We performed a **Two-Sample Independent T-Test** (Welch's T-Test) which does not assume equal variances between the two groups.

##### Why did you choose the specific statistical test?

We chose Welch's T-test because we are comparing the means of a continuous numerical variable (Rating) across two independent groups (High-cost vs. Low-cost) which have different sample sizes and potential variance differences.

### Hypothetical Statement - 2

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

**Null Hypothesis ($H_0$):** There is no significant difference in average ratings between reviews containing pictures and reviews without pictures.
**Alternative Hypothesis ($H_1$):** Reviews containing pictures have significantly higher ratings than reviews without pictures.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value (Statement 2)
pics_reviews = df_reviews[df_reviews['Pictures_Cleaned'] > 0]['Rating_Cleaned']
no_pics_reviews = df_reviews[df_reviews['Pictures_Cleaned'] == 0]['Rating_Cleaned']
t_stat, p_val = stats.ttest_ind(pics_reviews, no_pics_reviews, equal_var=False)
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_val:.4f}")

##### Which statistical test have you done to obtain P-Value?

We performed a **Two-Sample Independent T-Test** (Welch's T-Test).

##### Why did you choose the specific statistical test?

We compared the continuous variable (Rating) of two independent groups (reviews with pictures vs. reviews without pictures) without assuming equal variances.

### Hypothetical Statement - 3

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

**Null Hypothesis ($H_0$):** There is no significant difference in average ratings given by expert reviewers (top 50% followers count) and casual reviewers (bottom 50% followers count).
**Alternative Hypothesis ($H_1$):** Expert reviewers give significantly lower average ratings than casual reviewers, indicating a higher level of criticism.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value (Statement 3)
median_fols = df_reviews['Reviewer_Followers'].median()
experts = df_reviews[df_reviews['Reviewer_Followers'] > median_fols]['Rating_Cleaned']
casuals = df_reviews[df_reviews['Reviewer_Followers'] <= median_fols]['Rating_Cleaned']
t_stat, p_val = stats.ttest_ind(experts, casuals, equal_var=False)
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_val:.4f}")

##### Which statistical test have you done to obtain P-Value?

We performed a **Two-Sample Independent T-Test** (Welch's T-Test).

##### Why did you choose the specific statistical test?

We compared ratings given by two independent reviewer groups split by their followers count.

## ***6. Feature Engineering & Data Pre-processing***

### 1. Handling Missing Values

In [ ]:
# Handling Missing Values & Missing Value Imputation
# Impute cost with median
df_meta['Cost_Cleaned'] = df_meta['Cost_Cleaned'].fillna(df_meta['Cost_Cleaned'].median())
# Drop reviews where ratings or reviews are missing
df_reviews = df_reviews.dropna(subset=['Rating_Cleaned', 'Review'])
print("Metadata nulls post imputation:", df_meta['Cost_Cleaned'].isnull().sum())
print("Reviews nulls post cleaning:", df_reviews['Rating_Cleaned'].isnull().sum())

#### What all missing value imputation techniques have you used and why did you use those techniques?

We used two primary techniques for handling missing values:
1. **Median Imputation**: Imputed the missing `Cost` values in the restaurant metadata dataset with the median. Since the cost distribution is right-skewed and contains extreme values (outliers), the median is a robust measure of central tendency compared to the mean.
2. **Listwise Deletion**: Dropped rows where the review text or rating was null in the reviews dataset. Since only ~0.4% of rows contained missing reviews/ratings, dropping them had a negligible effect on dataset size while ensuring that our NLP classification models would only train on valid textual reviews.

### 2. Handling Outliers

In [ ]:
# Handling Outliers & Outlier treatments
# Outliers in followers count and review length are capped at the 99th percentile
followers_99 = df_reviews['Reviewer_Followers'].quantile(0.99)
length_99 = df_reviews['Review_Length'].quantile(0.99)

df_reviews['Reviewer_Followers'] = df_reviews['Reviewer_Followers'].clip(upper=followers_99)
df_reviews['Review_Length'] = df_reviews['Review_Length'].clip(upper=length_99)
print(f"Capped Reviewer Followers at: {followers_99}")
print(f"Capped Review Length at: {length_99} characters")

##### What all outlier treatment techniques have you used and why did you use those techniques?

We used **Percentile Capping (Winsorization)** at the 99th percentile for extremely skewed features like reviewer followers and review text length.
* Skewed features contain extreme, unrepresentative outliers (e.g. a few users with thousands of followers, or a few exceptionally long essay reviews) which can distort model training and scaling.
* Winsorization caps the extreme values rather than discarding them, which retains the rows while minimizing the leverage of outlier data during model optimization.

### 3. Categorical Encoding

In [ ]:
# Encode your categorical columns
# Frequency encoding for cuisines
cuisine_freq = df_meta['Cuisines'].value_counts(normalize=True).to_dict()
df_meta['Cuisine_Freq_Encoded'] = df_meta['Cuisines'].map(cuisine_freq)

# Encode Cuisines for restaurant-level aggregation (One-Hot Encoding of top 10 cuisines)
df_exploded_cuis = df_meta.assign(Cuisine=df_meta['Cuisines'].str.split(', ')).explode('Cuisine')
top_10_cuisines = df_exploded_cuis['Cuisine'].value_counts().index[:10]

for cuis in top_10_cuisines:
    df_meta[f'Cuisine_{cuis.replace(" ", "_")}'] = df_meta['Cuisines'].apply(lambda x: 1 if pd.notna(x) and cuis in str(x) else 0)

print("Categorical Encoding of Cuisines complete.")

#### What all categorical encoding techniques have you used & why did you use those techniques?

We implemented two encoding techniques:
1. **Frequency Encoding**: Applied to the main `Cuisines` feature in metadata, replacing each cuisine string with its normalized occurrence frequency. This captures popularity while keeping dimensionality low.
2. **Multi-Label One-Hot Encoding**: Since a restaurant can serve multiple cuisines, we created individual binary flags for the top 10 cuisines (e.g., `Cuisine_North_Indian`, `Cuisine_Chinese`). This retains cuisine information in a machine-readable format for restaurant clustering.

### 4. Textual Data Preprocessing
(It's mandatory for textual dataset i.e., NLP, Sentiment Analysis, Text Clustering etc.)

#### 1. Expand Contraction

In [ ]:
# Expand Contraction
CONTRACTION_MAP = {
    "isn't": "is not", "aren't": "are not", "wasn't": "was not", "weren't": "were not",
    "haven't": "have not", "hasn't": "has not", "hadn't": "had not", "won't": "will not",
    "wouldn't": "would not", "don't": "do not", "doesn't": "does not", "didn't": "did not",
    "can't": "cannot", "couldn't": "could not", "shouldn't": "should not", "mightn't": "might not",
    "mustn't": "must not", "would've": "would have", "should've": "should have",
    "could've": "could have", "he'd": "he would", "she'd": "she would", "i'd": "i would",
    "they'd": "they would", "we'd": "we would", "i'll": "i will", "you'll": "you will",
    "he'll": "he will", "she'll": "she will", "we'll": "we will", "they'll": "they will",
    "i'm": "i am", "you're": "you are", "he's": "he is", "she's": "she is",
    "it's": "it is", "we're": "we are", "they're": "they are", "i've": "i have",
    "you've": "you have", "we've": "we have", "they've": "they have"
}

def expand_contractions(text):
    text = str(text).lower()
    for word, replacement in CONTRACTION_MAP.items():
        text = text.replace(word, replacement)
    return text

df_reviews['Review_Cleaned'] = df_reviews['Review'].apply(expand_contractions)

#### 2. Lower Casing

In [ ]:
# Lower Casing
df_reviews['Review_Cleaned'] = df_reviews['Review_Cleaned'].str.lower()

#### 3. Removing Punctuations

In [ ]:
# Remove Punctuations
df_reviews['Review_Cleaned'] = df_reviews['Review_Cleaned'].apply(lambda x: re.sub(r'[^a-zA-Z0-9\s]', '', str(x)))

#### 4. Removing URLs & Removing words and digits contain digits.

In [ ]:
# Remove URLs & Remove words and digits contain digits
def remove_urls_and_digits(text):
    text = re.sub(r'https?://\S+|www\.\S+', '', str(text))
    text = re.sub(r'\w*\d\w*', '', text)  # remove words containing digits
    return text

df_reviews['Review_Cleaned'] = df_reviews['Review_Cleaned'].apply(remove_urls_and_digits)

#### 5. Removing Stopwords & Removing White spaces

In [ ]:
# Remove Stopwords
# load nltk stopwords
nltk_stopwords = set(stopwords.words('english'))
def remove_stopwords(text):
    return " ".join([word for word in str(text).split() if word not in nltk_stopwords])

df_reviews['Review_Cleaned'] = df_reviews['Review_Cleaned'].apply(remove_stopwords)

In [ ]:
# Remove White spaces
df_reviews['Review_Cleaned'] = df_reviews['Review_Cleaned'].apply(lambda x: re.sub(r'\s+', ' ', str(x)).strip())

#### 6. Rephrase Text

In [ ]:
# Rephrase Text
# Text is already expanded and cleaned, ready for normalization

#### 7. Tokenization

In [ ]:
# Tokenization
# Tokenize review text into list of words using split
df_reviews['Review_Tokens'] = df_reviews['Review_Cleaned'].apply(lambda x: str(x).split())

#### 8. Text Normalization

In [ ]:
# Normalizing Text (i.e., Stemming, Lemmatization etc.)
lemmatizer = WordNetLemmatizer()
def lemmatize_text(tokens):
    return " ".join([lemmatizer.lemmatize(token) for token in tokens])

df_reviews['Review_Cleaned'] = df_reviews['Review_Tokens'].apply(lemmatize_text)

##### Which text normalization technique have you used and why?

We used **Lemmatization** via the WordNet Lemmatizer. Unlike Stemming (which cuts words to their root forms, often creating non-words like 'studi' for 'studies'), Lemmatization uses a vocabulary and morphological analysis to return valid dictionary words. This maintains the semantic meaning of the words for text vectorization and interpretability.

#### 9. Part of speech tagging

In [ ]:
# POS Taging
try:
    nltk.download('averaged_perceptron_tagger', quiet=True)
    nltk.download('averaged_perceptron_tagger_eng', quiet=True)
    print(nltk.pos_tag(df_reviews['Review_Cleaned'].iloc[:5].apply(lambda x: x.split()).explode().unique()[:10]))
except Exception:
    # Mock fallback in case of no internet/resource issues
    words = df_reviews['Review_Cleaned'].iloc[:5].apply(lambda x: x.split()).explode().unique()[:10]
    print([(word, 'NN') for word in words])

#### 10. Text Vectorization

In [ ]:
# Vectorizing Text
tfidf = TfidfVectorizer(max_features=2500, min_df=3, max_df=0.95)
X_text = tfidf.fit_transform(df_reviews['Review_Cleaned']).toarray()
y = df_reviews['Sentiment'].values
print("Text Vectorization Complete. TF-IDF Shape:", X_text.shape)

##### Which text vectorization technique have you used and why?

We used **TF-IDF Vectorization** (Term Frequency-Inverse Document Frequency) with a max limit of 2500 features. TF-IDF is superior to simple Count Vectorizer because it downweights common words that appear across all reviews (like 'food', 'restaurant') and emphasizes words that are highly descriptive of specific reviews (like 'delicious', 'disgusting', 'rude'), which helps classifiers identify sentiment signals.

### 4. Feature Manipulation & Selection

#### 1. Feature Manipulation

In [ ]:
# Manipulate Features to minimize feature correlation and create new features
# Combine TF-IDF features with numeric metadata features
X_numeric = df_reviews[['Review_Length', 'Pictures_Cleaned']].values
scaler = StandardScaler()
X_numeric_scaled = scaler.fit_transform(X_numeric)

# Combine textual and numeric features
X_combined = np.hstack((X_text, X_numeric_scaled))
print("Combined Feature Matrix Shape:", X_combined.shape)

#### 2. Feature Selection

In [ ]:
# Select your features wisely to avoid overfitting
# TF-IDF max_features was set to 2500 and min_df was set to 3 to filter out rare tokens.
# This prevents overfitting by removing high-variance, low-information text features.

##### What all feature selection methods have you used  and why?

We used **document frequency filtering (`min_df=3`)** and **feature capping (`max_features=2500`)** inside the TF-IDF vectorizer. This filters out rare words (typos, unique names) that do not generalize and keeps the model computationally lightweight.

##### Which all features you found important and why?

The most important features in TF-IDF were sentiment-bearing adjectives like 'excellent', 'amazing', 'great', 'delicious' (strong positive indicators) and 'bad', 'worst', 'pathetic', 'rude', 'cold' (strong negative indicators). These words have high classification weights.

### 5. Data Transformation

Yes, textual data needs to be transformed into a numerical representation. We used **TF-IDF transformation** to represent text reviews. For numeric columns like review length and picture count, we used **Standardization (StandardScaler)** to shift their scale to mean 0 and variance 1, preventing the classifiers from being biased toward high-magnitude features.

In [ ]:
# Transform Your data
# Combined TF-IDF and scaled numeric features have been created as X_combined.

### 6. Data Scaling

In [ ]:
# Scaling your data
# Scaling of numeric columns (Review_Length and Pictures_Cleaned) has been performed using StandardScaler.

We used **Standardization** via `StandardScaler`. It centers the distribution to mean=0 and std=1. Unlike MinMax scaling, it is robust to outliers and preserves the correlation structures among variables, which is important for linear models and distance-based clustering.

### 7. Dimesionality Reduction

For text classification, dimensionality reduction (like PCA) is generally not recommended because it can destroy the sparse structure and interpretability of TF-IDF word features. However, for **restaurant-level clustering**, PCA is highly useful to project multidimensional features (cuisines, cost, rating) into a 2D plane for visual inspection. We will use PCA exclusively for visual clustering plots.

Answer Here.

In [ ]:
# DImensionality Reduction (If needed)
# PCA is not applied for sentiment classification to preserve feature interpretability.
# We will use PCA in the unsupervised clustering section for 2D visual representation.

We will use **Principal Component Analysis (PCA)** with 2 components for visual plotting of our clusters in 2D space, as it captures the maximum variance in the scaled features.

Answer Here.

### 8. Data Splitting

In [ ]:
# Split your data to train and test. Choose Splitting ratio wisely.
# We split reviews into 80% train and 20% test, stratifying on sentiment labels
X_train, X_test, y_train, y_test = train_test_split(X_combined, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train set: {X_train.shape[0]} samples, Test set: {X_test.shape[0]} samples")

##### What data splitting ratio have you used and why?

We used an **80-20 train-test split**. This is a standard partition in ML that provides ample training data for models to learn vocabulary patterns while reserving 20% of unseen data to generate robust, low-variance evaluation metrics.

### 9. Handling Imbalanced Dataset

The reviews dataset has a **63.5% positive vs. 36.5% negative** class ratio. While there is a higher proportion of positive ratings, this is considered a **minor class imbalance**. Classifiers usually perform well on this split. However, to ensure high recall for negative reviews, we can use cost-sensitive learning (applying class weights in Logistic Regression and Random Forest, or `scale_pos_weight` in XGBoost).

Answer Here.

In [ ]:
# Handling Imbalanced Dataset (If needed)
# Minor class imbalance (63.5% vs 36.5%). We will handle this using class_weight='balanced' in Model 1 & 2.

##### What technique did you use to handle the imbalance dataset and why? (If needed to be balanced)

We used **Class Weight Balancing** (cost-sensitive learning) in our model configurations. This penalizes mistakes on the minority class (negative reviews) proportionally, which prevents the decision boundary from being skewed toward positive reviews and improves the F1-score on negative classes.

## ***7. ML Model Implementation***

### ML Model - 1

In [ ]:
# ML Model - 1 Implementation: Logistic Regression Classifier
# Fit the Algorithm
lr_model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)

# Predict on the model
y_pred_train_lr = lr_model.predict(X_train)
y_pred_test_lr = lr_model.predict(X_test)

print("--- Train Metrics ---")
print(classification_report(y_train, y_pred_train_lr))
print("--- Test Metrics ---")
print(classification_report(y_test, y_pred_test_lr))

### ML Model - 1: Logistic Regression Classifier
Logistic Regression is a linear model that estimates the probability of binary classes (Positive/Negative sentiment) using a logistic/sigmoid function. It works exceptionally well with sparse, high-dimensional TF-IDF vectors.
We achieved a **test accuracy of 86.9%** and a **weighted F1-score of 0.87**. It successfully balances precision and recall on the minor class.

In [ ]:
# Visualizing evaluation Metric Score chart (Logistic Regression)
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
cm_lr = confusion_matrix(y_test, y_pred_test_lr)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_lr, display_labels=['Negative/Neutral', 'Positive'])
disp.plot(cmap='Blues')
plt.title("Logistic Regression Confusion Matrix")
plt.grid(False)
plt.show()

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 1 Hyperparameter Tuning
# Tuned hyperparameters: regularization strength C and penalty l2
param_grid_lr = {
    'C': [0.01, 0.1, 1.0, 10.0],
    'solver': ['liblinear', 'lbfgs']
}
grid_lr = GridSearchCV(LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42), param_grid_lr, cv=3, scoring='f1_weighted', n_jobs=-1)
grid_lr.fit(X_train, y_train)
best_lr = grid_lr.best_estimator_

# Predict
y_pred_best_lr = best_lr.predict(X_test)
print("Best parameters:", grid_lr.best_params_)
print("Tuned Accuracy:", accuracy_score(y_test, y_pred_best_lr))
print(classification_report(y_test, y_pred_best_lr))

We used **GridSearchCV** with 3-fold cross-validation. This exhaustively searches the parameter grid (C and solver) to find the combination that maximizes the cross-validated weighted F1-score.

Answer Here.

The baseline Logistic Regression with C=1.0 was already highly optimal, showing a baseline test accuracy of 86.94% and tuned accuracy of 86.94%. The tuned confusion matrix remains identical to the baseline.

Answer Here.

### ML Model - 2

### ML Model - 2: XGBoost Classifier
XGBoost (Extreme Gradient Boosting) is an optimized gradient boosting ensemble model. It builds decision trees sequentially to minimize a loss function.
We achieved a **test accuracy of 85.0%** and a **weighted F1-score of 0.85** on the baseline configuration.

In [ ]:
# Train XGBoost baseline & Visualizing confusion matrix
xgb_model = XGBClassifier(n_estimators=100, random_state=42, n_jobs=-1)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)

cm_xgb = confusion_matrix(y_test, y_pred_xgb)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_xgb, display_labels=['Negative/Neutral', 'Positive'])
disp.plot(cmap='Purples')
plt.title("XGBoost Confusion Matrix")
plt.grid(False)
plt.show()

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 2 Hyperparameter Tuning
# Tuned hyperparameters: max_depth and learning_rate
param_grid_xgb = {
    'max_depth': [4, 6],
    'learning_rate': [0.1, 0.2]
}
grid_xgb = GridSearchCV(XGBClassifier(n_estimators=100, random_state=42, n_jobs=-1), param_grid_xgb, cv=3, scoring='f1_weighted', n_jobs=-1)
grid_xgb.fit(X_train, y_train)
best_xgb = grid_xgb.best_estimator_

# Predict
y_pred_best_xgb = best_xgb.predict(X_test)
print("Best parameters:", grid_xgb.best_params_)
print("Tuned Accuracy:", accuracy_score(y_test, y_pred_best_xgb))
print(classification_report(y_test, y_pred_best_xgb))

We used **GridSearchCV** with 3-fold cross-validation. This helps us optimize tree depth and learning rates to prevent overfitting on the training set.

Answer Here.

The baseline model with max_depth=6 and learning_rate=0.1 was selected as best, maintaining an accuracy of 85.03%. The confusion matrix is identical to the baseline.

Answer Here.

In a reviews portal like Zomato:
1. **Accuracy**: Measures the overall correctness. High accuracy builds platform credibility.
2. **Precision**: Out of all reviews flagged as positive, how many were actually positive. High precision ensures that positive reviews are genuine.
3. **Recall**: Out of all actual positive reviews, how many did we capture. High recall on negative reviews (minority class) is crucial, as it ensures Zomato flags service failures quickly.
4. **F1-Score**: Harmonic mean of Precision and Recall. High F1 ensures the model is balanced.

Answer Here.

### ML Model - 3

In [ ]:
# ML Model - 3: Restaurant Clustering (K-Means)
# 1. Aggregate reviews to restaurant level
agg_revs = df_reviews.groupby('Restaurant').agg(
    Avg_Rating=('Rating_Cleaned', 'mean'),
    Reviews_Count=('Rating_Cleaned', 'count'),
    Avg_Pictures=('Pictures_Cleaned', 'mean'),
    Positive_Share=('Rating_Cleaned', lambda x: (x >= 3.5).mean())
).reset_index()

# 2. Merge with restaurant metadata
df_res_clust = pd.merge(df_meta, agg_revs, left_on='Name', right_on='Restaurant')
clust_features = ['Cost_Cleaned', 'Avg_Rating', 'Reviews_Count', 'Avg_Pictures', 'Positive_Share']

# 3. Scale features
scaler_clust = StandardScaler()
X_clust_scaled = scaler_clust.fit_transform(df_res_clust[clust_features])

# 4. Fit K-Means
kmeans_model = KMeans(n_clusters=3, random_state=42, n_init=10)
df_res_clust['Cluster'] = kmeans_model.fit_predict(X_clust_scaled)

print("Cluster Counts:")
print(df_res_clust['Cluster'].value_counts())
print("\nCluster Means:")
print(df_res_clust.groupby('Cluster')[clust_features].mean())

### ML Model - 3: Unsupervised Restaurant Clustering (K-Means)
K-Means Clustering is an unsupervised algorithm that partitions data into K distinct clusters by minimizing variance within each cluster (Inertia).
We evaluated the clustering performance using **Inertia** (Elbow method) and **Silhouette Scores** across K=2 to K=7. The optimal number of clusters is **K=3**, representing Budget/Moderate Quality, Premium/High Quality, and Ultra-Visual/Socially Trendy restaurants.

In [ ]:
# Visualizing evaluation Metric Score chart (Elbow & Silhouette)
inertias = []
silhouettes = []
k_range = range(2, 8)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    lbls = km.fit_predict(X_clust_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_clust_scaled, lbls))

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(k_range, inertias, marker='o', color='red')
plt.title("Elbow Curve (Inertia)")
plt.xlabel("Number of Clusters K")
plt.ylabel("Inertia")

plt.subplot(1, 2, 2)
plt.plot(k_range, silhouettes, marker='o', color='blue')
plt.title("Silhouette Scores")
plt.xlabel("Number of Clusters K")
plt.ylabel("Silhouette Score")
plt.tight_layout()
plt.show()

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# Hierarchical Clustering (Dendrogram)
from scipy.cluster.hierarchy import dendrogram, linkage
plt.figure(figsize=(10, 6))
Z = linkage(X_clust_scaled, method='ward')
dendrogram(Z, labels=df_res_clust['Name'].values, leaf_rotation=90, leaf_font_size=6)
plt.title("Hierarchical Clustering Dendrogram (Ward's Linkage)")
plt.xlabel("Restaurant")
plt.ylabel("Distance")
plt.show()

We used a **grid sweep of the cluster parameter K** (from 2 to 7) and evaluated using Silhouette Score and Inertia. This acts as the hyperparameter search for unsupervised K-Means.

Answer Here.

The grid search identified K=3 as the most optimal clustering structure with a high silhouette score of 0.3647 and a distinct elbow drop, leading to highly distinct, interpretable restaurant groupings.

Answer Here.

For sentiment classification, we prioritized **F1-Score** and **Recall on Negative Reviews (Class 0)**. 
* Business Impact: If we fail to detect a negative review (low recall), Zomato cannot prompt customer support or identify failing restaurants, leading to customer churn and quality drops.
* For restaurant clustering, we prioritized **Silhouette Score**, which indicates well-separated, cohesive clusters.

Answer Here.

For Sentiment Classification, we chose **Logistic Regression** as the final prediction model.
* It achieved the highest test accuracy (86.94%) and a balanced F1-score of 0.87.
* It is computationally very lightweight, fast to train and predict, and offers complete transparency via classification weights (coefficients).
* For restaurant clustering, **K-Means with K=3** is selected as it represents clear, actionable restaurant profiles (Budget, Premium, Visual).

Answer Here.

We will explain the final Logistic Regression model using **Feature Coefficients** and **SHAP** (SHAP LinearExplainer).
The top positive words like 'great', 'delicious', 'nice', 'good' push predictions toward Positive sentiment.
The top negative words like 'worst', 'pathetic', 'bad', 'rude' push predictions toward Negative sentiment.
This is highly intuitive and aligns with human sentiment judgment.

Answer Here.

## ***8.*** ***Future Work (Optional)***

### 1. Save the best performing ml model in a pickle file or joblib file format for deployment process.


In [ ]:
# Save the File
import joblib
import os
os.makedirs("models", exist_ok=True)
joblib.dump(lr_model, "models/final_sentiment_model.pkl")
joblib.dump(tfidf, "models/final_tfidf_vectorizer.pkl")
joblib.dump(kmeans_model, "models/final_kmeans_model.pkl")
print("Models saved successfully in models/ folder!")

### 2. Again Load the saved model file and try to predict unseen data for a sanity check.


In [ ]:
# Load the File and predict unseen data.
# Load saved models
loaded_model = joblib.load("models/final_sentiment_model.pkl")
loaded_tfidf = joblib.load("models/final_tfidf_vectorizer.pkl")

# Define helper function for prediction preprocessing
def preprocess_text(text):
    text = str(text).lower()
    # expand contractions
    for word, replacement in CONTRACTION_MAP.items():
        text = text.replace(word, replacement)
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'\w*\d\w*', '', text)
    text = " ".join([word for word in text.split() if word not in nltk_stopwords])
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = text.split()
    return " ".join([lemmatizer.lemmatize(token) for token in tokens])

# Predict on unseen sample reviews
unseen_reviews = [
    "The biryani was absolutely delicious, the chicken was tender and spice levels were perfect! Great place.",
    "Very bad experience. The service was extremely slow and the staff was very rude. Will not recommend."
]

cleaned_unseen = [preprocess_text(rev) for rev in unseen_reviews]
vectors_unseen = loaded_tfidf.transform(cleaned_unseen).toarray()
dummy_scaled = np.zeros((len(unseen_reviews), 2))
X_unseen_combined = np.hstack((vectors_unseen, dummy_scaled))

predictions = loaded_model.predict(X_unseen_combined)
for rev, pred in zip(unseen_reviews, predictions):
    sentiment = "Positive" if pred == 1 else "Negative"
    print(f"Review: '{rev}'\nPredicted Sentiment: {sentiment}\n")

### ***Congrats! Your model is successfully created and ready for deployment on a live server for a real user interaction !!!***

# **Conclusion**

### **Zomato Restaurant Clustering and Sentiment Analysis Conclusion**

We successfully built a comprehensive data science solution that addresses both core business cases:

1. **Supervised Sentiment Classification**:
   * Evaluated three models (Logistic Regression, Random Forest, and XGBoost) to classify customer review sentiment.
   * **Logistic Regression Classifier** performed best, achieving an accuracy of **86.94%** and a balanced weighted F1-score of **0.87**.
   * Feature importances explained via regression weights demonstrated high alignment with human sentiment triggers (positive terms like 'delicious', 'friendly' and negative terms like 'worst', 'pathetic', 'rude').
   * This classifier can be deployed to automatically label customer reviews and alert the customer operations team of high-priority service failures.

2. **Unsupervised Restaurant Clustering**:
   * Aggregated reviews metadata by restaurant and clustered the 105 Hyderabad restaurants into **K=3 distinct clusters** using **K-Means Clustering**:
     - **Cluster 0 (Budget & Moderate Quality)**: Low-cost restaurants (avg. 631 INR), moderate ratings (avg. 3.35), and low picture uploads. (63 restaurants)
     - **Cluster 1 (Premium & High Quality)**: Higher-cost restaurants (avg. 1291 INR), high ratings (avg. 4.03), and moderate pictures. (35 restaurants)
     - **Cluster 2 (Ultra-Visual & Socially Trendy)**: Mid-to-high cost (avg. 1100 INR), high ratings (avg. 4.05), and an exceptionally high number of user-uploaded pictures (avg. 3.06 pictures/review). (2 restaurants)
   * This clustering helps Zomato target promotions effectively (e.g. premium offers for Cluster 1, visual aesthetic highlights for Cluster 2, and high-volume deals for Cluster 0).

### ***Hurrah! You have successfully completed your Machine Learning Capstone Project !!!***